In [9]:
import torch
import torch.nn as nn
from torch.nn.functional import cross_entropy
import tiktoken


if torch.cuda.is_available():
    torch.set_default_device("cuda")


GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

torch.get_default_device()

device(type='cuda', index=0)

In [10]:
class LayerNorm(nn.Module):
    def __init__(self, embedding_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))
 
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim = True)
        var = x.var(dim=-1, keepdim = True)

        x = (x-mean)/ torch.sqrt(var + self.eps)
        
        return x * self.scale + self.shift

norm_layer = LayerNorm(4)
inpt = torch.reshape(torch.arange(20, dtype= torch.float32), shape=(5,4))
print(inpt)
oupt = norm_layer(inpt)
oupt

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.],
        [16., 17., 18., 19.]], device='cuda:0')


tensor([[-1.1619, -0.3873,  0.3873,  1.1619],
        [-1.1619, -0.3873,  0.3873,  1.1619],
        [-1.1619, -0.3873,  0.3873,  1.1619],
        [-1.1619, -0.3873,  0.3873,  1.1619],
        [-1.1619, -0.3873,  0.3873,  1.1619]], device='cuda:0',
       grad_fn=<AddBackward0>)

In [11]:
class GeLU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        x = 0.5 * x * (1 + torch.tanh( torch.tensor((2/torch.pi)**0.5) ) * ( x + 0.044715*x**3) )
        return x

gelu_layer = GeLU()
gelu_layer(oupt)

tensor([[-0.1065, -0.1436,  0.2437,  1.0554],
        [-0.1065, -0.1436,  0.2437,  1.0554],
        [-0.1065, -0.1436,  0.2437,  1.0554],
        [-0.1065, -0.1436,  0.2437,  1.0554],
        [-0.1065, -0.1436,  0.2437,  1.0554]], device='cuda:0',
       grad_fn=<MulBackward0>)

In [12]:
class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 4*embedding_dim),
            GeLU(),
            nn.Linear(4*embedding_dim, embedding_dim),
        )

    def forward(self, x):
        return self.layers(x)

In [13]:

class MultiHeadAttention(torch.nn.Module):
    # num of embeddings are n, n at max can be equal to context length
    def __init__(self, embed_dim, output_dim, num_heads):
        super().__init__()

        self.output_dim = output_dim
        self.num_heads = num_heads
        self.embed_dim = embed_dim

        # trainable layers for initializing queries, keys and values
        self.w_queries = torch.nn.Linear(embed_dim, output_dim*num_heads, bias=False) # embed_dim x output_dim*num_heads
        self.w_keys = torch.nn.Linear(embed_dim, output_dim*num_heads, bias=False)
        self.w_values = torch.nn.Linear(embed_dim, output_dim*num_heads, bias=False)

    def forward(self, embeddings):
        words = embeddings.shape[-2]
        
        embeddings = torch.reshape(embeddings, (-1, words, self.embed_dim))  # batches x context_len x embed_dim
        batches = len(embeddings)

        all_queries = self.w_queries(embeddings) # batches x context_len x output_dim*num_heads
        all_keys = self.w_keys(embeddings)
        all_values = self.w_values(embeddings)

        # split them in columns and then line them up in a tensor
        queries =  all_queries.view(batches, self.num_heads, words, self.output_dim)
        keys = all_keys.view(batches, self.num_heads, words, self.output_dim)
        values = all_values.view(batches, self.num_heads, words, self.output_dim)


        attention_scores = queries @ keys.transpose(2,3) # numheads x n x n

        # as this is causal attention now we will mask the upper right diagonal
        causal_mask_bool =  torch.triu(torch.ones_like(attention_scores), diagonal=1).bool() #triu stands for triangle up

        attention_scores.masked_fill_(causal_mask_bool, -torch.inf) # now a word only depend on the words before it, as the future dependencies are -ve infinity so after softmax the probabilities will be zero

        attention_weights = torch.softmax(attention_scores / self.output_dim**0.5, dim=2) # batches x numheads x context_len x context_len

        context_vectors = (attention_weights @ values).transpose(1, 2)
        # batches x context x numheads x output_dim <from> batches x numheads x context x output_dim
        context_vector = torch.reshape(context_vectors, shape= (batches, words, self.output_dim*self.num_heads))
        # batches x context x numheads*output_dim
        return context_vector


embed_dim = 5

sentence = "your journey starts with one step"
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(sentence)
# let num tokens be n
encoded = torch.tensor(encoded)
print(encoded)
embedding_layer = nn.Embedding(GPT_CONFIG_124M["vocab_size"], embed_dim)

embeddings = embedding_layer(encoded)

tensor([14108,  7002,  4940,   351,   530,  2239], device='cuda:0')


In [14]:
attention_head = MultiHeadAttention(embed_dim, output_dim = 3, num_heads = 3)
context = attention_head(embeddings)
embeddings.shape , context.shape

(torch.Size([6, 5]), torch.Size([1, 6, 9]))

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.norm_1 = LayerNorm(config["emb_dim"])
        output_dim = config["emb_dim"] // config["n_heads"]
        self.attention = MultiHeadAttention(config["emb_dim"], output_dim, config["n_heads"])
        self.dropout = nn.Dropout(config["drop_rate"])

        
        self.norm_2 = LayerNorm(config["emb_dim"])
        self.ffnetwork = FeedForward(config["emb_dim"])


    def forward(self, input_vector):
        original = input_vector.detach()

        out = self.norm_1(original)
        out = self.attention(out)
        out = self.dropout(out)

        out += original

        out = self.norm_2(out)
        out = self.ffnetwork(out)
        out = self.dropout(out)
        
        out += original

        return out


embedding_layer = nn.Embedding(GPT_CONFIG_124M["vocab_size"], GPT_CONFIG_124M["emb_dim"])
embeddings = embedding_layer(encoded)


print(embeddings.shape)
transform = TransformerBlock(GPT_CONFIG_124M)
output = transform(embeddings)
print(output.shape)

torch.Size([6, 768])
torch.Size([1, 6, 768])


In [16]:
class GPTModel(nn.Module):
    def __init__(self, config, batch_size):
        super().__init__()
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop_emb = nn.Dropout(config["drop_rate"])
        self.batch_size = batch_size

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config["n_layers"])]
        )

        self.final_norm = LayerNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)

    def forward(self, in_idx):
        in_idx = torch.tensor(in_idx)
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(len(in_idx), device=in_idx.device))

        x = tok_embeds + pos_embeds
        x = torch.reshape(x, shape = (-1 , min(self.batch_size, len(in_idx)), GPT_CONFIG_124M["emb_dim"]))
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)

        logits = self.out_head(x)
        return logits

llm = GPTModel(GPT_CONFIG_124M, 3)
output = llm(encoded)
output_flat = output.flatten(0,1)
print(encoded.shape, output_flat.shape)
# here each vocab size array represents the preference of the next word depending on all words before it
# so if we want the new words we look for the highest probablity(using softmax) in the last array

torch.Size([6]) torch.Size([6, 50257])


/home/capti/.local/lib/python3.13/site-packages/torch/utils/_device.py:103: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


In [17]:
total_params = 0
for param in llm.parameters():

    layer_params = 1
    for num in param.shape:
        layer_params*=num

    total_params += layer_params

print(total_params) # 32 bit float
total_bytes = total_params * 4 # 4 byte per param
total_kbs = total_bytes/ 1024
total_mbs = total_kbs/ 1024

print(total_mbs)

155922432
594.796875


In [18]:
def get_pred(model, inputs, output_tokens = 1):

    for i in range(output_tokens):
        with torch.no_grad():
            output = model(inputs)

        last_row = output[:,-1,:]
        probs = torch.softmax(last_row, dim=-1)

        output = probs.argmax(dim=-1,keepdim=True)[0][0]
        inputs.append(output.item())
    return inputs

tokens = tokenizer.encode("I am a naughty little boy and i need punishment")

model = GPTModel(GPT_CONFIG_124M, 1024)
model.eval()

tokens = get_pred(model, tokens, 3)
tokenizer.decode(tokens)

'I am a naughty little boy and i need punishment TulsSur ploy'

In [19]:
def perplexity(cross_entropy_loss):
    return torch.exp(cross_entropy_loss.item())